In [1]:
!pip install -q transformers peft datasets accelerate

In [2]:
!pip install --upgrade torchao

In [3]:
import sys
import os
from google.colab import drive

# 1. Monta il Drive
drive.mount('/content/drive')

# 2. Definisci i percorsi principali
BASE_DRIVE = '/content/drive/MyDrive/DeepLearning'
SRC_DIR = os.path.join(BASE_DRIVE, 'src')

# 3. Aggiungi la cartella 'src' ai percorsi di sistema di Python
if SRC_DIR not in sys.path:
    sys.path.append(SRC_DIR)
    print(f"Directory {SRC_DIR} aggiunta al path di sistema.")

# 4. Ora puoi importare normalmente come se fossero librerie installate
from dataset import CLEVRDataset
from models import MultimodalCoT

import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer
from peft import LoraConfig, get_peft_model
from tqdm import tqdm

# Impostiamo il device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Setup completato. Device in uso: {device}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Directory /content/drive/MyDrive/DeepLearning/src aggiunta al path di sistema.


✅ Setup completato. Device in uso: cuda


In [4]:
from torchvision import transforms

# 1. Inizializziamo il Tokenizer del T5 (o del modello LLM base che stai usando)
MODEL_NAME = "t5-base" # Cambialo se usi t5-base o un altro modello
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# 2. Definiamo la pipeline di trasformazione per le immagini
# (Usa quella esatta che richiede il tuo vision_encoder)
transform = transforms.Compose([
    transforms.Resize((384, 384)), # Adatta alla dimensione richiesta dal tuo encoder
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("✅ Tokenizer e Transform pronti.")

✅ Tokenizer e Transform pronti.


In [5]:
# Percorsi assoluti verso il tuo ecosistema "blindato"
BASE_DRIVE = '/content/drive/MyDrive/DeepLearning'
TRAIN_INDEX = os.path.join(BASE_DRIVE, 'indexes/train_index.json')
TRAIN_IMG_DIR = os.path.join(BASE_DRIVE, 'data/processed/images/train')

# Inizializziamo il Dataset
train_dataset = CLEVRDataset(
    index_path=TRAIN_INDEX,
    img_dir=TRAIN_IMG_DIR,
    tokenizer=tokenizer,
    transform=transform,
    stage='stage2'
)

# --- CREIAMO IL CUSTOM COLLATOR ---
def custom_collate_fn(batch):
    import torch
    return {
        # Impila le immagini in un unico tensore [batch_size, channels, height, width]
        'pixel_values': torch.stack([item['pixel_values'] for item in batch]),

        # Lascia i testi come semplici liste di stringhe (saranno processati dal tokenizer nella Cella 5)
        'input_text': [item['input_text'] for item in batch],
        'target_text': [item['target_text'] for item in batch],

        # Lascia i dati grezzi come semplice lista di dizionari (ignorati da PyTorch)
        'raw_item': [item['raw_item'] for item in batch]
    }

# Creiamo il DataLoader inserendo la nostra funzione personalizzata
BATCH_SIZE = 8
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,
    collate_fn=custom_collate_fn # <--- Il fix è qui!
)

print(f"✅ DataLoader pronto. Campioni totali: {len(train_dataset)} | Batch totali: {len(train_loader)}")

✅ DataLoader pronto. Campioni totali: 15000 | Batch totali: 1875


In [6]:
# Percorsi assoluti verso il tuo ecosistema "blindato"
BASE_DRIVE = '/content/drive/MyDrive/DeepLearning'
VAL_INDEX = os.path.join(BASE_DRIVE, 'indexes/val_index.json')
VAL_IMG_DIR = os.path.join(BASE_DRIVE, 'data/processed/images/val')

# Inizializziamo il Dataset
val_dataset = CLEVRDataset(
    index_path=VAL_INDEX,
    img_dir=VAL_IMG_DIR,
    tokenizer=tokenizer,
    transform=transform,
    stage='stage2'
)

# Creiamo il DataLoader inserendo la nostra funzione personalizzata
BATCH_SIZE = 8
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,
    collate_fn=custom_collate_fn # <--- Il fix è qui!
)

print(f"✅ DataLoader pronto. Campioni totali: {len(val_dataset)} | Batch totali: {len(val_loader)}")

✅ DataLoader pronto. Campioni totali: 1000 | Batch totali: 125


In [7]:
# 1. Inizializza l'architettura completa
model = MultimodalCoT().to(device)

# 2. Configurazione LoRA per il modello linguistico interno
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q", "v"], # Verifica che questi siano i layer corretti per T5
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_2_SEQ_LM"
)

# 3. Innestiamo LoRA nel sottomodello LLM
model.llm = get_peft_model(model.llm, lora_config)

# Stampiamo i parametri addestrabili per conferma
model.llm.print_trainable_parameters()

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

trainable params: 884,736 || all params: 223,788,288 || trainable%: 0.3953


In [8]:
from torch.optim import AdamW
import torch.amp
import os
import torch
from tqdm import tqdm

# IPERPARAMETRI OTTIMIZZATI
EPOCHS = 5
LEARNING_RATE = 5e-4
ACCUMULATION_STEPS = 4 # Aggiorna i pesi ogni 4 batch (simula una batch size 4 volte più grande)
ADAPTER_DIR = os.path.join(BASE_DRIVE, 'lora_adapter_stage2') # Assicurati che BASE_DRIVE sia definito
os.makedirs(ADAPTER_DIR, exist_ok=True)

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
scaler = torch.amp.GradScaler('cuda') # Il "Motore" per la velocità a 16-bit

# Inizializziamo la variabile per tracciare la migliore validation loss
best_val_loss = float('inf')

print("🚀 Inizio Addestramento e Validazione (Stage 2)...")

for epoch in range(EPOCHS):
    # ==========================================
    # 1. FASE DI TRAINING
    # ==========================================
    model.train()
    total_train_loss = 0
    optimizer.zero_grad() # Resettiamo all'inizio dell'epoca

    progress_bar = tqdm(train_loader, desc=f"Train Epoca {epoch+1}/{EPOCHS}")

    for step, batch in enumerate(progress_bar):
        pixel_values = batch['pixel_values'].to(device)

        # Padding dinamico: input lungo (512), target corto (16)
        inputs = tokenizer(batch['input_text'], return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
        labels_encoding = tokenizer(batch['target_text'], return_tensors="pt", padding=True, truncation=True, max_length=5)

        labels = labels_encoding.input_ids
        # Ignoriamo il padding nella Loss
        labels[labels == tokenizer.pad_token_id] = -100
        labels = labels.to(device)

        # Forward pass veloce
        with torch.amp.autocast('cuda'):
            outputs = model(
                pixel_values=pixel_values,
                input_ids=inputs.input_ids,
                attention_mask=inputs.attention_mask,
                labels=labels
            )
            # Dividiamo la loss per i passi di accumulo
            loss = outputs.loss / ACCUMULATION_STEPS

        # Backward pass scalato
        scaler.scale(loss).backward()

        # 💥 CORREZIONE BUG ACCUMULO: Taglio e aggiornamento pesi avvengono SOLO qui, ogni 4 batch!
        if (step + 1) % ACCUMULATION_STEPS == 0 or (step + 1) == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        # Sommiamo per la loss media dell'epoca
        total_train_loss += (loss.item() * ACCUMULATION_STEPS)
        progress_bar.set_postfix({'loss': f"{(loss.item() * ACCUMULATION_STEPS):.4f}"})

    avg_train_loss = total_train_loss / len(train_loader)

    # ==========================================
    # 2. FASE DI VALIDATION (EARLY STOPPING)
    # ==========================================
    model.eval()
    total_val_loss = 0
    val_progress_bar = tqdm(val_loader, desc=f"Validazione Epoca {epoch+1}/{EPOCHS}")

    with torch.no_grad(): # Disattiviamo il calcolo dei gradienti per risparmiare memoria
        for batch in val_progress_bar:
            pixel_values = batch['pixel_values'].to(device)
            inputs = tokenizer(batch['input_text'], return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)

            labels_encoding = tokenizer(batch['target_text'], return_tensors="pt", padding=True, truncation=True, max_length=16)
            labels = labels_encoding.input_ids
            labels[labels == tokenizer.pad_token_id] = -100
            labels = labels.to(device)

            # Inferenza veloce (nessun backward qui!)
            with torch.amp.autocast('cuda'):
                outputs = model(
                    pixel_values=pixel_values,
                    input_ids=inputs.input_ids,
                    attention_mask=inputs.attention_mask,
                    labels=labels
                )
                loss = outputs.loss

            total_val_loss += loss.item()
            val_progress_bar.set_postfix({'val_loss': f"{loss.item():.4f}"})

    avg_val_loss = total_val_loss / len(val_loader)

    print(f"\n🏁 Fine Epoca {epoch+1}")
    print(f"📉 Train Loss Media: {avg_train_loss:.4f} | 📊 Val Loss Media: {avg_val_loss:.4f}")

    # ==========================================
    # 3. SALVATAGGIO OTTIMIZZATO (Best Model)
    # ==========================================
    if avg_val_loss < best_val_loss:
        print(f"🌟 Miglioramento! Val Loss scesa da {best_val_loss:.4f} a {avg_val_loss:.4f}. Salvataggio pesi in corso...\n")
        best_val_loss = avg_val_loss

        # Salviamo la configurazione vincente
        model.llm.save_pretrained(ADAPTER_DIR)
        torch.save(model.projector.state_dict(), os.path.join(ADAPTER_DIR, 'projector.pth'))
        torch.save(model.cross_attention.state_dict(), os.path.join(ADAPTER_DIR, 'cross_attn.pth'))
    else:
        print(f"⚠️ Nessun miglioramento. I pesi non vengono sovrascritti (Best Val Loss rimane {best_val_loss:.4f}).\n")

    print("-" * 60)

🚀 Inizio Addestramento e Validazione (Stage 2)...


Validazione Epoca 1/5: 100%|██████████| 125/125 [06:24<00:00,  3.07s/it, val_loss=0.6206]



🏁 Fine Epoca 1
📉 Train Loss Media: nan | 📊 Val Loss Media: 0.5666
🌟 Miglioramento! Val Loss scesa da inf a 0.5666. Salvataggio pesi in corso...

------------------------------------------------------------


Validazione Epoca 2/5: 100%|██████████| 125/125 [00:33<00:00,  3.70it/s, val_loss=0.9464]



🏁 Fine Epoca 2
📉 Train Loss Media: 0.5559 | 📊 Val Loss Media: 0.5171
🌟 Miglioramento! Val Loss scesa da 0.5666 a 0.5171. Salvataggio pesi in corso...

------------------------------------------------------------


...


🏁 Fine Epoca 3
📉 Train Loss Media: nan | 📊 Val Loss Media: 0.4903
🌟 Miglioramento! Val Loss scesa da 0.5171 a 0.4903. Salvataggio pesi in corso...

------------------------------------------------------------


Validazione Epoca 4/5: 100%|██████████| 125/125 [00:32<00:00,  3.88it/s, val_loss=0.5569]



🏁 Fine Epoca 4
📉 Train Loss Media: 0.5349 | 📊 Val Loss Media: 0.4917
⚠️ Nessun miglioramento. I pesi non vengono sovrascritti (Best Val Loss rimane 0.4903).

------------------------------------------------------------


Validazione Epoca 5/5: 100%|██████████| 125/125 [00:31<00:00,  3.97it/s, val_loss=0.5641]


🏁 Fine Epoca 5
📉 Train Loss Media: 0.5091 | 📊 Val Loss Media: 0.5000
⚠️ Nessun miglioramento. I pesi non vengono sovrascritti (Best Val Loss rimane 0.4903).

------------------------------------------------------------


In [9]:
# Definiamo la cartella di output finale
ADAPTER_DIR = os.path.join(BASE_DRIVE, 'lora_adapter_stage2')
os.makedirs(ADAPTER_DIR, exist_ok=True)

# 1. Salviamo il cervello linguistico (LoRA e Tokenizer)
model.llm.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

# 2. SALVIAMO GLI OCCHI E IL RAGIONAMENTO (I moduli Custom)
custom_weights = {
    'projector': model.projector.state_dict(),
    'cross_attention': model.cross_attention.state_dict(),
    'layer_norm': model.layer_norm.state_dict()
}
torch.save(custom_weights, os.path.join(ADAPTER_DIR, 'custom_modules.pth'))

print(f"🎉 Modello e moduli visivi salvati con successo in: {ADAPTER_DIR}")

🎉 Modello e moduli visivi salvati con successo in: /content/drive/MyDrive/DeepLearning/lora_adapter_stage2
